# 3 Govern Usage, Cost, and Limits

**Purpose:**
Finish the path from Section 0: see what your calls cost, cap how fast they can arrive, cap how much they can spend, and write client code that behaves well when the gateway says no.

**What we're doing**

We read the usage that Section 2's calls left behind, put a price on your vLLM Resource, set a request rate limit and a monthly wallet budget, push requests until the gateway rejects them, and record exactly what a client sees. Then we write a small client helper that reacts correctly to each kind of rejection.

**Why we're doing it**

* Section 2 covered the *access* half of the LLM Gateway: who may call which model. This notebook covers the *cost and visibility* half that the Orientation promised.
* Limits are only useful if the people behind them can see them work, and if client code can tell "slow down" from "you are out of money" from "the model is down"
* A governed endpoint is one where someone has capped what it can cost

**What to look for**

* The calls from Section 2 in the **AI Usage** dashboard, and why their cost reads zero until a price is set
* The gateway rejecting a request, and the status code, headers, and body your client receives
* Whether a rate-limit rejection and a budget rejection look different to a client

**Output**

A rate limit and a wallet budget that you have watched reject requests, and a client helper `chat_with_limits()` that retries, stops, or falls back as appropriate.

> **Before you start:** Sections 3.4 to 3.6 change gateway settings, which needs a Klique administrator (or an administrator's help), as in the Section 2b notebook. Section 3.6 can also lock a user out of the gateway for the rest of the month. Read its warning first, and use a throwaway user or project.

## 3.1 The Four Controls

Everything in this notebook is one of four controls. The first three are settings; the fourth is how you see them working.

| Control | What it limits | Where an administrator sets it | Docs |
|---|---|---|---|
| **Request rate limit** | How many requests a user can send in a rolling window | LLM Gateway settings: a gateway-wide default, a per-**Profile** limit, and per-user or per-group overrides inside a profile. The API reference also describes a per-**Resource** limit, and says the profile-level one is deprecated (see 3.5). | [Request Rate Limit](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway#request-rate-limit) |
| **Wallet budget** | How many **dollars** a project or user can spend per calendar month, across every model they use | The AI Quotas settings page: a default per project and per user, plus allocated budgets for specific ones | [AI Quotas](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_ai_quotas) |
| **Pricing** | What one million tokens cost on a Resource. It turns token counts into dollars. | Each Resource's settings: input, output, cache read, and cache write cost per 1M tokens | [Resources](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway#resources) |
| **AI Usage** | Nothing. It shows tokens, cost, and latency, by model, user, provider, user group, or project. | The AI Usage tab of the Admin Dashboard | [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage) |

Two facts shape the rest of the notebook:

* **A budget is money, not tokens.** Without a price on a Resource, its usage costs zero, and a dollar budget never fills.
* **The two limits are independent.** The docs say a request is rejected once it exceeds *either* the rate limit or the budget.

## 3.2 Load the Token and the Gateway URL

This is the setup from Section 2, plus three helpers:

* `send()` sends one chat completion and returns the raw response. Unlike `chat()` in Section 2, it does **not** raise on an HTTP error, because in this notebook the errors are what we are studying.
* `describe()` prints a response's status, the headers that look limit-related, and the start of the body.
* `fingerprint()` boils a rejection down to its status code and error label, so two rejections can be compared.

Set `PROFILE` and `GATEWAY_MODEL` to the same values as in the Section 2a notebook. The token is shown by its last four characters only.

In [1]:
import sys
import time
import json
import random
sys.path.insert(0, "..")
from config import KLIQUE_GATEWAY_TOKEN as token, KLIQUE_LLM_GATEWAY_URL as gateway_url
import requests

PROFILE = "frank-test"                 # Your LLM Gateway profile
GATEWAY_MODEL = "vllm-qwen-franktest"  # Your Section 1 endpoint's name in that profile
FALLBACK_MODEL = None                  # Optional: a second alias in the profile, used in 3.8

print(f"LLM Gateway:   {gateway_url}")
print(f"Profile:       {PROFILE}")
print(f"Gateway model: {GATEWAY_MODEL}")
print(f"Token:         ...{token[-4:]}")


def send(prompt, model_name=None, max_tokens=64, timeout=60):
    """Send one chat completion through the gateway and return the raw Response."""
    return requests.post(
        f"{gateway_url}/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {token}",
            "X-ClearML-Profile": PROFILE,
        },
        json={
            "model": model_name or GATEWAY_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        },
        timeout=timeout,
    )


def describe(resp):
    """Print what a client sees: status, limit-related headers, and the start of the body."""
    print(f"HTTP {resp.status_code} {resp.reason}")
    keywords = ("limit", "retry", "remaining", "reset", "quota", "budget")
    for name, value in resp.headers.items():
        if any(k in name.lower() for k in keywords):
            print(f"  header {name}: {value}")
    print(f"  body: {resp.text[:400]}")


def fingerprint(resp):
    """Reduce a rejection to (status code, error code or type) so two rejections can be compared."""
    label = None
    try:
        body = resp.json()
    except ValueError:
        body = None
    if isinstance(body, dict):
        err = body.get("error")
        if isinstance(err, dict):
            label = err.get("code") or err.get("type")
        elif isinstance(err, str):
            label = err
    return (resp.status_code, label)

LLM Gateway:   https://task-router-allegro-master-server.services.internal.allegro.ai/llm
Profile:       frank-test
Gateway model: vllm-qwen-franktest
Token:         ...ypZo


## 3.3 Read Your Usage

Every call through the gateway is metered. Send five short requests, and keep your own running total of the tokens they used, so you can check the dashboard against it.

In [2]:
PROMPTS = [
    "In one sentence, what is a GPU?",
    "In one sentence, what is a token in a language model?",
    "In one sentence, what does an API gateway do?",
    "In one sentence, what is a rate limit?",
    "In one sentence, what is a budget?",
]

total_in = total_out = 0
for prompt in PROMPTS:
    r = send(prompt, max_tokens=64)
    if not r.ok:
        describe(r)
        break
    usage = r.json()["usage"]
    total_in += usage["prompt_tokens"]
    total_out += usage["completion_tokens"]

print(f"Calls sent:        {len(PROMPTS)}")
print(f"Prompt tokens:     {total_in}")
print(f"Completion tokens: {total_out}")

Calls sent:        5
Prompt tokens:     92
Completion tokens: 320


Now open the **AI Usage** tab of the Admin Dashboard:

1. Set the source to **Gateway**, so external usage is left out.
2. Set the time range to cover the last hour.
3. Read the **Tokens** cards: **In** and **Out** should be close to the totals above, plus any other calls you made in Section 2.
4. Read the **Usage Breakdown** table. Group it by **model**, then by **user**, then by **project**.

The breakdown groups usage by model, user, provider, user group, or project. The docs do not list profile alias as a grouping, so look for your calls under the model that served them.

Now read the **Cost** card. It is probably **$0.00**. The docs explain why: "usage from resources without pricing shows a cost of zero."

> **Checkpoint:** you found your calls in the dashboard, and the token totals roughly match the cell above. If **Gateway** shows nothing, the dashboard may be filtering to a different user or time range.

<!-- screenshot: images/ai_usage_tokens.png (AI Usage: Tokens cards and the Usage Breakdown table grouped by model) -->

## 3.4 Put a Price on Your vLLM Resource

*An administrator does this step.*

Your own GPU has no per-token bill, so Klique has no cost to read from your vLLM Resource. You supply one. In the LLM Gateway settings, edit the Resource that points at your Section 1 endpoint and fill in **Pricing**: input and output cost per 1M tokens. Cache read and cache write are optional, and the docs say blank cache costs price cached tokens at zero.

The prices below are **teaching values**, not a recommendation. They are deliberately large, so a handful of calls produces a visible cost. Enter the same two numbers in the Resource and in the cell.

> **A price change is not retroactive.** The docs say each request's cost is fixed when it is recorded, so the calls from 3.3 stay at $0.00. Send new calls after you save the price.

In [3]:
PRICE_PER_1M_INPUT = 50.0    # dollars per 1M input tokens: teaching value, set the same on the Resource
PRICE_PER_1M_OUTPUT = 150.0  # dollars per 1M output tokens: teaching value, set the same on the Resource

run_in = run_out = 0
for prompt in PROMPTS:
    r = send(prompt, max_tokens=64)
    if not r.ok:
        describe(r)
        break
    usage = r.json()["usage"]
    run_in += usage["prompt_tokens"]
    run_out += usage["completion_tokens"]

expected_cost = run_in / 1e6 * PRICE_PER_1M_INPUT + run_out / 1e6 * PRICE_PER_1M_OUTPUT
print(f"Prompt tokens:     {run_in}")
print(f"Completion tokens: {run_out}")
print(f"Expected cost:     ${expected_cost:.6f}")

Prompt tokens:     92
Completion tokens: 320
Expected cost:     $0.052600


Open **AI Usage** again, switch the charts to **Cost**, and compare the **Cost** card with *Expected cost* above. They should agree for the calls sent after the price was saved.

**Optional: compare with an external model.** If your profile has an external model whose Resource also has pricing, send it the same five prompts (change `model_name` in `send()`), and compare the two cost figures and the **Avg Latency** card. Compare them only if both Resources have prices. A Resource without pricing always reads zero, which makes a free-looking comparison that means nothing.

> **Checkpoint:** the dashboard's **Cost** card is above zero and close to *Expected cost*.

## 3.5 Set a Request Rate Limit

*An administrator does this step.*

A rate limit counts requests, not tokens or dollars, over a rolling window. It is evaluated from the most specific setting to the least specific: a **user override**, then a **group override**, then the **profile's own limit**, then the **gateway-wide default**.

For this exercise, set a limit on your profile only, so nobody else is affected:

1. Open the LLM Gateway settings and edit your profile.
2. In **Rate Limit**, set **Requests** to `5` and **Window in seconds** to `60`.
3. Save, and set `WINDOW_SECONDS` in the cell below to match.

**The docs disagree about where this limit lives.** The LLM Gateway settings page describes a profile-level **Rate Limit**, as above. The [API reference](https://klique.ai/docs/latest/docs/references/api/routers) marks a profile's `rate_limit` as *deprecated*: "the gateway does not apply" it, because "it limits requests per resource." It describes a rate limit on the **Resource** instead, shared by every alias that reaches the Resource, with an `action` of `reject` or `queue`. With `reject`, an over-limit request gets a `429`. With `queue`, it waits for a slot and is never rejected.

So if the loop below never gets a rejection, ask your administrator to set the rate limit on the Resource (as in the API reference) instead of the profile.

The next cell sends tiny requests in a loop until the gateway pushes back, then prints what a client receives. The API reference gives `429` for a rejected over-limit request on a Resource. It does not give the error body or any retry header. The same reference also mentions a `429` for an exhausted wallet, in its description of Routed Resource rules (`on_quota_limit` set to `reject`), so the two rejections may share a status code. This cell records what yours actually does.

In [4]:
WINDOW_SECONDS = 60   # same as "Window in seconds" on the profile
MAX_REQUESTS = 20     # stop here if nothing is rejected

rate_limit_response = None
for i in range(1, MAX_REQUESTS + 1):
    r = send("Say OK.", max_tokens=8)
    print(f"request {i:>2}: HTTP {r.status_code}")
    if not r.ok:
        rate_limit_response = r
        break

if rate_limit_response is None:
    print(f"\nNo rejection after {MAX_REQUESTS} requests. Check the profile's Rate Limit settings.")
else:
    print("\nWhat the client received:")
    describe(rate_limit_response)
    rate_limit_fp = fingerprint(rate_limit_response)
    print(f"\nFingerprint: {rate_limit_fp}")

request  1: HTTP 200
request  2: HTTP 200
request  3: HTTP 200
request  4: HTTP 200
request  5: HTTP 200
request  6: HTTP 200
request  7: HTTP 200
request  8: HTTP 200
request  9: HTTP 200
request 10: HTTP 200
request 11: HTTP 200
request 12: HTTP 200
request 13: HTTP 200
request 14: HTTP 200
request 15: HTTP 200
request 16: HTTP 200
request 17: HTTP 200
request 18: HTTP 200
request 19: HTTP 200
request 20: HTTP 200

No rejection after 20 requests. Check the profile's Rate Limit settings.


Write down three things: the **status code**, any **retry-related header**, and the **error body**. Your client code in 3.8 depends on them.

Now wait for the window to pass and confirm the gateway lets you back in:

In [5]:
if rate_limit_response is not None:
    print(f"Waiting {WINDOW_SECONDS + 5} seconds...")
    time.sleep(WINDOW_SECONDS + 5)
    r = send("Say OK.", max_tokens=8)
    print(f"After the window: HTTP {r.status_code}")

> **Checkpoint:** the loop ended with a rejection, and the request after the wait succeeded. A rate-limit rejection is temporary: waiting is the right response. Did you get a `429`, as the API reference says?

**Put the rate limit back before 3.6.** Clear the limit you set, or raise it well above what the next section sends. Otherwise the budget loop keeps hitting the rate limit.

## 3.6 Set a Wallet Budget

*An administrator does this step.*

A wallet budget caps dollars per calendar month for a project or a user. Because it counts dollars, it only works for Resources that have pricing, so 3.4 comes first.

> **Warning: this can lock someone out for the rest of the month.** The docs say that once an entity's spend crosses its budget, Klique rejects further requests from it "until the next calendar month, unless enforcement is turned off." Do not put a tiny budget on your own everyday user. Use a throwaway user or project, and know how to undo it: turn **Enforce wallet budgets** off.

On the AI Quotas settings page:

1. Turn on **Enforce wallet budgets**. When it is off, spend is tracked and requests are never blocked.
2. Click **Allocate budget**, choose the **throwaway** user or project, and set **Limit ($ / month)** low enough to cross in a few dozen calls. With the teaching prices from 3.4 and short requests, a limit of `0.05` is a reasonable start.
3. Run the cell below as that user, or with a token that belongs to them.

**Two docs pages disagree about pricing and budgets.** The AI Quotas page says wallet budgets are "calculated using the pricing configured for the Resources." The LLM Gateway settings page says reported spend "is separate from wallet budget enforcement and does not reduce a wallet's balance." This notebook does not assume which is right. The cell below finds out: if your calls never get rejected, the second reading applies to your setup, and you should tell your administrator.

Watch your own spend as you go: the user's **Personal wallet** is shown under **AI Spend**, in **Settings** on the **Profile** tab.

In [ ]:
MAX_CALLS = 60

budget_response = None
calls_ok = 0
for i in range(1, MAX_CALLS + 1):
    r = send("Write two sentences about why teams cap AI spend.", max_tokens=200)
    if r.ok:
        calls_ok += 1
        print(f"call {i:>2}: HTTP {r.status_code}  (total tokens {r.json()['usage']['total_tokens']})")
        continue

    fp = fingerprint(r)
    if "rate_limit_fp" in globals() and fp == rate_limit_fp and rate_limit_response is not None:
        # Same rejection as 3.5: the rate limit is still on. Wait it out and carry on.
        print(f"call {i:>2}: looks like the rate limit from 3.5. Waiting {WINDOW_SECONDS + 5}s.")
        time.sleep(WINDOW_SECONDS + 5)
        continue

    budget_response = r
    break

if budget_response is None:
    print(f"\nNo rejection after {MAX_CALLS} calls. See the note above about the two docs pages.")
else:
    print(f"\nRejected after {calls_ok} successful calls. What the client received:")
    describe(budget_response)
    budget_fp = fingerprint(budget_response)
    print(f"\nFingerprint: {budget_fp}")

Compare the two rejections you recorded.

In [ ]:
if "rate_limit_fp" in globals() and "budget_fp" in globals():
    print(f"Rate limit: {rate_limit_fp}")
    print(f"Budget:     {budget_fp}")
    if rate_limit_fp == budget_fp:
        print("\nThey look the same to a client. Code cannot tell 'wait' from 'out of money' by status and error label alone.")
    else:
        print("\nThey differ. A client can tell them apart by status and error label.")
else:
    print("Run 3.5 and 3.6 first, so both rejections are recorded.")

Then read the budget in the WebApp: the **Spent this month** and **Limit ($ / month)** columns of the Allocated Budgets table, and the Personal wallet under **AI Spend**.

> **Checkpoint:** a budget rejection happened, and you know whether it differs from the rate-limit rejection.

**Undo it now.** Turn **Enforce wallet budgets** off, or delete the allocated budget, before you move on. Confirm with one call:

In [ ]:
r = send("Say OK.", max_tokens=8)
print(f"After undoing the budget: HTTP {r.status_code}")

## 3.7 What a Client Should Do With Each Rejection

The right reaction depends on *why* the request was rejected:

| What happened | Is waiting useful? | Client action |
|---|---|---|
| **Rate limit** | Yes: it clears when the rolling window passes | Wait and retry, with exponential backoff and a cap on attempts |
| **Wallet budget** | No: it lasts until the next calendar month, or until an administrator changes it | Stop. Report it clearly. Do **not** retry. |
| **Model unreachable** (`502`, `503`, `504`, or a timeout) | Sometimes | Retry once or twice, then try another alias if you have one |
| **`401` or `404`** | No: the token, profile, or alias is wrong | Stop and fix the configuration |

Why not fall back to another alias when the budget runs out? The budget belongs to the project or user, not to one Resource, so a second *paid* alias is rejected too. The API reference notes one exception: a Routed Resource rule can pass over to the next rule when the wallet is exhausted (`on_quota_limit` set to `next_rule`, the default), and only a *free* target can take over. Fallback in client code is for an unreachable model.

> **The gateway does not fall back for you.** Section 2b's Routed Resources choose a Resource by request attributes such as max output tokens. The docs do not describe routing around a model that is down. Fallback here is code in your client.

The helper in 3.8 depends on 3.5 and 3.6: it recognizes a rejection by the fingerprint you recorded. If the two fingerprints were identical, the helper cannot tell them apart. It then treats the rejection as a possible rate limit, retries a bounded number of times, and gives up with a clear message.

## 3.8 A Client Helper That Behaves Under Limits

`chat_with_limits()` tries each model in `models` in order:

* **Success:** returns the parsed reply.
* **Rate limit** (or an ambiguous rejection): waits and retries the same model. The wait is the `Retry-After` header if the gateway sent a number of seconds, otherwise exponential backoff with a little random jitter.
* **Budget:** raises `GovernanceError` immediately.
* **Unreachable model** (a connection error, a timeout, or `502`, `503`, `504`): retries once, then moves to the next model.
* **Anything else** (`401`, `404`, and so on): raises the HTTP error.

In [ ]:
UPSTREAM_STATUSES = {502, 503, 504}


class GovernanceError(Exception):
    """The gateway refused the request in a way that retrying will not fix."""


def classify(resp):
    """Label a response: ok, rate_limit, budget, ambiguous, upstream, or other."""
    if resp.ok:
        return "ok"
    fp = fingerprint(resp)
    rl = globals().get("rate_limit_fp")
    bd = globals().get("budget_fp")
    if rl and bd and rl == bd and fp == rl:
        return "ambiguous"
    if rl and fp == rl:
        return "rate_limit"
    if bd and fp == bd:
        return "budget"
    if resp.status_code in UPSTREAM_STATUSES:
        return "upstream"
    return "other"


def wait_time(resp, attempt, base_delay):
    """Seconds to wait: Retry-After if it is a number, else exponential backoff with jitter."""
    header = resp.headers.get("Retry-After") if resp is not None else None
    if header is not None:
        try:
            return float(header)
        except ValueError:
            pass
    return base_delay * (2 ** attempt) + random.uniform(0, 1)


def chat_with_limits(prompt, models, max_tokens=256, max_retries=4, base_delay=2.0):
    """Send a chat completion, retrying or falling back as the rejection allows."""
    for model_name in models:
        attempt = 0
        while True:
            resp = None
            try:
                resp = send(prompt, model_name, max_tokens)
                kind = classify(resp)
            except requests.RequestException:
                kind = "upstream"

            if kind == "ok":
                return resp.json()
            if kind == "budget":
                raise GovernanceError("Wallet budget reached. Retrying will not help until it resets or is raised.")
            if kind == "other":
                resp.raise_for_status()
                raise GovernanceError(f"Unexpected response: HTTP {resp.status_code}")

            limit = max_retries if kind in ("rate_limit", "ambiguous") else 1
            if attempt >= limit:
                if kind == "upstream":
                    print(f"'{model_name}' is unreachable. Trying the next model, if any.")
                    break
                raise GovernanceError(f"Still rejected after {attempt} retries ({kind}).")

            delay = wait_time(resp, attempt, base_delay)
            print(f"{kind}: waiting {delay:.1f}s before retry {attempt + 1}")
            time.sleep(delay)
            attempt += 1

    raise GovernanceError("No model in the list answered.")

Use it exactly like `chat()` from Section 2, with a list of aliases:

In [ ]:
models = [GATEWAY_MODEL] + ([FALLBACK_MODEL] if FALLBACK_MODEL else [])

try:
    reply = chat_with_limits("In one sentence, why do clients need retry logic?", models)
    print(reply["choices"][0]["message"]["content"])
    print(f"\nServed by: {reply['model']}")
except GovernanceError as e:
    print(f"Stopped: {e}")

**Try it under pressure.** Repeat 3.5's low rate limit, and call `chat_with_limits()` in a loop of ten requests. It should print waits and still finish. Then try the fallback: set `FALLBACK_MODEL` to a second alias in your profile, stop your Section 1 instance, and call the helper. The gateway cannot reach the stopped endpoint, so the helper should report the model unreachable and answer from the fallback. Relaunch your instance afterwards if you want to keep going.

> **Checkpoint:** under a rate limit the helper waits and succeeds. With a stopped instance and a fallback alias, it answers from the fallback.

## 3.9 Put Everything Back

| Setting | Undo |
|---|---|
| Resource **Pricing** (3.4) | Keep it if you want cost on your dashboards, or clear it |
| Profile **Rate Limit** (3.5) | Clear **Requests** and **Window in seconds** to inherit the gateway default |
| **Enforce wallet budgets** and the allocated budget (3.6) | Turn enforcement off, or delete the allocated budget |
| Stopped Section 1 instance (3.8) | Relaunch it, or leave it stopped to release the GPU |

## Troubleshooting

| Symptom | Likely cause |
|---|---|
| **Cost** reads $0.00 | The Resource that served the call has no **Pricing**, or the calls were sent before the price was saved. A price change is not retroactive. |
| The AI Usage tab is missing | AI Usage is part of the Admin Dashboard, so you may need an administrator. |
| 3.5 never rejects | The limit was saved on a profile, which the API reference says the gateway does not apply. Set it on the Resource. Or the Resource's action is `queue`, so over-limit requests wait instead of failing. Or the limit sits on a user or group override that does not apply to you. |
| 3.6 never rejects | **Enforce wallet budgets** is off, the Resource has no pricing, the budget is on a different user or project, or the second reading of the docs applies (see 3.6). |
| Every call is rejected after 3.6 | The budget is still on. Turn **Enforce wallet budgets** off. |
| `chat_with_limits()` raises "Still rejected" | The limit is longer than the retries cover. Raise `max_retries` or wait out the window. |
| `404` with `model_not_found` | The alias is not in your profile. See Section 2's troubleshooting. |

## Wrap-Up

You have now followed the whole path from the Orientation:

```
Model → vLLM Model Deployment → Model Endpoint → AI Application Gateway → LLM Gateway
 └──────────────── Section 1 ──────────────────────────────────────┘      └ Sections 2 and 3 ┘
```

You deployed a model, routed traffic to it, priced it, watched its usage, and capped what it can cost.

## Related Docs

- [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway)
- [LLM Gateway settings: rate limits, Resources, and Profiles](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway)
- [AI Quotas: wallet budgets](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_ai_quotas)
- [AI Cost and Rate Limit Control](https://klique.ai/docs/latest/docs/getting_started/token_quota_control)
- [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)
- [Tokenomics](https://klique.ai/docs/latest/docs/governance_engine/tokenomics)